# 05 — Optional research control: baseline gradient accumulation

This is **not required** for the assignment. It compares the reversible physical batch against a standard model reaching the same effective batch through gradient accumulation.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_URL = "https://github.com/JoeIndyGit/era-v5-session-13-reversible-llm-lab.git"
REPO_DIR = Path("/content/era-v5-session-13-reversible-llm-lab")
if not (Path.cwd() / "src").exists():
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
sys.path.insert(0, str(Path.cwd()))
print("repo root:", Path.cwd())


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)


In [ ]:
from pathlib import Path
import os, sys, json
ROOT=Path.cwd()
if not (ROOT/"src").exists():
    candidates=[p.parent for p in Path("/content").glob("**/src") if p.is_dir()]
    if candidates:
        ROOT=candidates[0]; os.chdir(ROOT)
sys.path.insert(0,str(Path.cwd()))
print("repo root:",Path.cwd())

In [ ]:
from src.evidence import load_config
import json,math
from src.model import ModelConfig
from src.train import run_accumulation_control
cfg=load_config(); base_cfg=ModelConfig(**json.load(open("configs/model_baseline.json")))
bp=json.load(open("results/baseline_batch_probe.json")); rp=json.load(open("results/reversible_batch_probe.json"))
target=rp["largest_stable_batch"]; limit=bp["largest_stable_batch"]
divisors=[d for d in range(1,limit+1) if target%d==0]
physical=max(divisors); accum=target//physical
print({"reversible_physical_batch":target,"baseline_physical_batch":physical,"gradient_accumulation_steps":accum,"matched_effective_batch":physical*accum})

In [ ]:
result=run_accumulation_control(cfg,base_cfg,"baseline_matched_effective_batch",physical_batch=physical,grad_accum_steps=accum)
result

Use this as an **extra control**, not as a replacement for any of the three required 50M-token runs.